<a href="https://colab.research.google.com/github/PaatriickC/CSCI164Search/blob/main/csci164problemsolving.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import heapq

In [ ]:
# General tile sliding puzzle setup for NxN grids
class TilePuzzle:
    def __init__(self, size):
        self.size = size
        self.goal = list(range(1, size * size)) + [0]

    def actions(self, state):
        index = state.index(0)
        row, col = divmod(index, self.size)
        moves = []
        if row > 0: moves.append(-self.size)  # Up
        if row < self.size - 1: moves.append(self.size)  # Down
        if col > 0: moves.append(-1)  # Left
        if col < self.size - 1: moves.append(1)  # Right
        return moves

    def result(self, state, action):
        new_state = state[:]
        index = state.index(0)
        swap_with = index + action
        new_state[index], new_state[swap_with] = new_state[swap_with], new_state[index]
        return new_state

    def goal_test(self, state):
        return state == self.goal

    def print_state(self, state):
        for i in range(0, len(state), self.size):
            print(state[i:i+self.size])
        print()

    def manhattan_distance(self, state):
        distance = 0
        for i, val in enumerate(state):
            if val == 0:
                continue
            goal_index = self.goal.index(val)
            x1, y1 = divmod(i, self.size)
            x2, y2 = divmod(goal_index, self.size)
            distance += abs(x1 - x2) + abs(y1 - y2)
        return distance

    def out_of_place(self, state):
        return sum(1 for i in range(len(state)) if state[i] != 0 and state[i] != self.goal[i])


In [ ]:
import random

def random_walk(puzzle, steps):
    state = puzzle.goal[:]
    path = []
    for _ in range(steps):
        actions = puzzle.actions(state)
        action = random.choice(actions)
        new_state = puzzle.result(state, action)
        path.append(new_state)
        state = new_state
    return state, path


In [ ]:
from collections import deque
import heapq

class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost

    def __lt__(self, other):
        return self.path_cost < other.path_cost

    def solution(self):
        path = []
        node = self
        while node.parent is not None:
            path.append(node.state)
            node = node.parent
        path.reverse()
        return path


In [ ]:
def bfs(puzzle, initial):
    frontier = deque([Node(initial)])
    explored = set()
    nodes_expanded = 0

    while frontier:
        node = frontier.popleft()
        nodes_expanded += 1
        if puzzle.goal_test(node.state):
            return node.solution(), nodes_expanded

        explored.add(tuple(node.state))

        for action in puzzle.actions(node.state):
            child_state = puzzle.result(node.state, action)
            if tuple(child_state) not in explored:
                frontier.append(Node(child_state, node, action, node.path_cost + 1))

    return None, nodes_expanded


In [ ]:
def astar(puzzle, initial, heuristic_fn):
    frontier = []
    start = Node(initial)
    heapq.heappush(frontier, (heuristic_fn(initial), start))
    explored = set()
    nodes_expanded = 0

    while frontier:
        _, node = heapq.heappop(frontier)
        nodes_expanded += 1
        if puzzle.goal_test(node.state):
            return node.solution(), nodes_expanded

        explored.add(tuple(node.state))

        for action in puzzle.actions(node.state):
            child_state = puzzle.result(node.state, action)
            if tuple(child_state) not in explored:
                cost = node.path_cost + 1
                h = heuristic_fn(child_state)
                heapq.heappush(frontier, (cost + h, Node(child_state, node, action, cost)))

    return None, nodes_expanded


In [ ]:
def run_experiment(puzzle, steps_list):
    results = []
    for steps in steps_list:
        for i in range(3):
            initial_state, _ = random_walk(puzzle, steps)

            print(f"Problem {steps} steps, instance {i+1}:")
            puzzle.print_state(initial_state)

            sol_bfs, bfs_nodes = bfs(puzzle, initial_state)
            sol_astar_manhattan, astar_m_nodes = astar(puzzle, initial_state, puzzle.manhattan_distance)
            sol_astar_oop, astar_oop_nodes = astar(puzzle, initial_state, puzzle.out_of_place)

            results.append({
                "steps": steps,
                "instance": i + 1,
                "start_state": initial_state,
                "length_bfs": len(sol_bfs),
                "nodes_bfs": bfs_nodes,
                "length_astar_manhattan": len(sol_astar_manhattan),
                "nodes_astar_manhattan": astar_m_nodes,
                "length_astar_oop": len(sol_astar_oop),
                "nodes_astar_oop": astar_oop_nodes
            })
    return results


In [ ]:
steps_list = [5, 10, 20, 40, 80]

# For 3x3
puzzle3 = TilePuzzle(3)
results3 = run_experiment(puzzle3, steps_list)

# For 4x4 (note: large sizes can make BFS infeasible)
puzzle4 = TilePuzzle(4)
results4 = run_experiment(puzzle4, steps_list)


Problem 5 steps, instance 1:
[1, 2, 3]
[4, 5, 0]
[7, 8, 6]

Problem 5 steps, instance 2:
[1, 2, 3]
[0, 4, 6]
[7, 5, 8]

Problem 5 steps, instance 3:
[1, 2, 3]
[4, 8, 5]
[7, 0, 6]

Problem 10 steps, instance 1:
[1, 3, 6]
[4, 2, 8]
[7, 5, 0]

Problem 10 steps, instance 2:
[0, 2, 3]
[1, 4, 6]
[7, 5, 8]

Problem 10 steps, instance 3:
[1, 2, 3]
[5, 0, 6]
[4, 7, 8]

Problem 20 steps, instance 1:
[1, 5, 0]
[8, 3, 2]
[4, 7, 6]

Problem 20 steps, instance 2:
[2, 3, 6]
[1, 0, 8]
[4, 5, 7]

Problem 20 steps, instance 3:
[1, 2, 3]
[7, 0, 5]
[8, 4, 6]

Problem 40 steps, instance 1:
[0, 5, 3]
[1, 2, 8]
[4, 7, 6]

Problem 40 steps, instance 2:
[1, 3, 5]
[4, 2, 6]
[0, 7, 8]

Problem 40 steps, instance 3:
[2, 3, 5]
[1, 0, 4]
[7, 8, 6]

Problem 80 steps, instance 1:
[1, 7, 8]
[5, 0, 6]
[3, 4, 2]

Problem 80 steps, instance 2:
[1, 8, 5]
[6, 0, 4]
[7, 2, 3]

Problem 80 steps, instance 3:
[7, 5, 0]
[4, 1, 2]
[8, 6, 3]

Problem 5 steps, instance 1:
[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 0]
[13, 14, 15, 12]



Could not run up to 80 steps for 4x4 due to insufficient RAM

In [ ]:
import pandas as pd

df3 = pd.DataFrame(results3)
df4 = pd.DataFrame(results4)

print("3x3 Puzzle Results:")
display(df3)

print("4x4 Puzzle Results:")
display(df4)
